# VPINN inverse 1D — version “article scientifique”

Ce notebook étend la version précédente pour produire des résultats beaucoup plus solides et présentables :

- reconstruction détaillée d'un cas de référence,
- étude de sensibilité au bruit,
- sweep sur `k_source`,
- répétitions sur plusieurs seeds,
- agrégation statistique,
- figures prêtes pour un rapport / article,
- sauvegarde systématique des résultats (`csv`, `json`, `png`).

## Idée centrale

On considère le problème inverse 1D :

\[
T''(x) + f(x) = 0,\qquad x\in(0,1),\qquad T(0)=T(1)=0,
\]

où l'on observe seulement des mesures bruitées de `T(x)` et l'on cherche à reconstruire la source `f(x)`.

La stratégie retenue est :

1. **Phase 1** : ajuster `T(x)` aux observations avec régularisation douce ;
2. **Phase 2** : reconstruire une initialisation de `f(x)` par solve faible direct ;
3. **Phase 3** : raffinement conjoint avec une loss **VPINN faible** ;
4. **Études expérimentales** : multi-seeds, bruit, complexité spectrale.

## Remarque pratique

Ce notebook contient deux niveaux de charge de calcul :

- un **mode showcase** pour un entraînement poussé sur un cas,
- un **mode benchmark** pour répéter de nombreuses expériences.

Si vous êtes sur CPU, commencez avec la configuration par défaut puis augmentez le budget si nécessaire.


In [ ]:

from __future__ import annotations

import gc
import json
import math
import os
import random
import time
from dataclasses import dataclass, asdict
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_GLOBAL)

torch.set_default_dtype(torch.float32)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT_DIR = "./results_vpinn_inverse_1d_article"
FIG_DIR = os.path.join(ROOT_DIR, "figures")
TABLE_DIR = os.path.join(ROOT_DIR, "tables")
RAW_DIR = os.path.join(ROOT_DIR, "raw")
for d in [ROOT_DIR, FIG_DIR, TABLE_DIR, RAW_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {DEVICE}")
print(f"Output root     : {ROOT_DIR}")

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

@dataclass
class ProblemConfig:
    n_obs: int = 25
    n_eval: int = 600
    n_quad: int = 256
    noise_rel: float = 0.01

@dataclass
class ModelConfig:
    k_test: int = 20
    k_source: int = 3
    ff_k: int = 8
    hidden: tuple[int, ...] = (64, 64, 64)

@dataclass
class TrainConfig:
    p1_epochs: int = 4000
    p1_lr: float = 2e-3
    p1_w_data: float = 1e3
    p1_w_h1: float = 1e-5
    p1_w_h2: float = 5e-4
    p2_ridge: float = 1e-6
    p2_ridge_power: int = 2
    p3_epochs: int = 10000
    p3_lr_T: float = 5e-4
    p3_lr_f: float = 2e-3
    p3_w_data: float = 1e3
    p3_w_weak: float = 2e2
    p3_w_anchor: float = 1e1
    p3_w_f_reg: float = 1e-6
    p3_w_h2: float = 1e-4
    p3_grad_clip: float = 1.0
    use_lbfgs: bool = True
    lbfgs_steps: int = 200
    lbfgs_lr: float = 0.4
    lbfgs_max_iter_per_step: int = 20

@dataclass
class BenchmarkConfig:
    seeds: tuple[int, ...] = (0, 1, 2, 3, 4)
    noise_grid: tuple[float, ...] = (0.00, 0.005, 0.01, 0.02, 0.05)
    k_source_grid: tuple[int, ...] = (2, 3, 4, 5, 6)
    sweep_p1_epochs: int = 1500
    sweep_p3_epochs: int = 3000
    sweep_lbfgs_steps: int = 40
    run_showcase: bool = True
    run_noise_study: bool = True
    run_k_source_sweep: bool = True
    run_full_grid: bool = True

PROB = ProblemConfig()
MODEL = ModelConfig()
TRAIN = TrainConfig()
BENCH = BenchmarkConfig()

print("ProblemConfig:", asdict(PROB))
print("ModelConfig  :", asdict(MODEL))
print("TrainConfig  :", asdict(TRAIN))
print("BenchmarkCfg :", asdict(BENCH))


In [ ]:

def f_true_np(x: np.ndarray) -> np.ndarray:
    return np.sin(np.pi * x) + 0.5 * np.sin(3.0 * np.pi * x)

def T_true_np(x: np.ndarray) -> np.ndarray:
    return np.sin(np.pi * x) / np.pi**2 + 0.5 * np.sin(3.0 * np.pi * x) / (9.0 * np.pi**2)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_dataset(problem_cfg: ProblemConfig, seed: int) -> dict[str, Any]:
    set_seed(seed)
    x_obs_np = np.linspace(0.05, 0.95, problem_cfg.n_obs, dtype=np.float32)
    T_clean_np = T_true_np(x_obs_np).astype(np.float32)
    noise_amp = float(problem_cfg.noise_rel * np.max(np.abs(T_clean_np)))
    T_obs_np = T_clean_np + noise_amp * np.random.randn(problem_cfg.n_obs).astype(np.float32)
    x_eval_np = np.linspace(0.0, 1.0, problem_cfg.n_eval, dtype=np.float32)
    data = {
        "x_obs_np": x_obs_np,
        "T_clean_np": T_clean_np,
        "T_obs_np": T_obs_np,
        "noise_amp": noise_amp,
        "noise_floor": float(noise_amp**2),
        "x_eval_np": x_eval_np,
        "f_true_eval_np": f_true_np(x_eval_np).astype(np.float32),
        "T_true_eval_np": T_true_np(x_eval_np).astype(np.float32),
        "x_obs": torch.tensor(x_obs_np, device=DEVICE).unsqueeze(1),
        "T_obs": torch.tensor(T_obs_np, device=DEVICE).unsqueeze(1),
        "x_eval": torch.tensor(x_eval_np, device=DEVICE).unsqueeze(1),
    }
    return data

data_demo = make_dataset(PROB, seed=0)

plt.figure(figsize=(8, 4))
plt.plot(data_demo["x_eval_np"], data_demo["T_true_eval_np"], label="T true", lw=2.5)
plt.scatter(data_demo["x_obs_np"], data_demo["T_obs_np"], label="observations", zorder=5)
plt.title(f"Synthetic observations | noise_rel={PROB.noise_rel:.1%}")
plt.xlabel("x")
plt.ylabel("T(x)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

def gauss_legendre_01(n: int, device: torch.device):
    xg, wg = np.polynomial.legendre.leggauss(n)
    xg = 0.5 * (xg + 1.0)
    wg = 0.5 * wg
    xq = torch.tensor(xg, dtype=torch.float32, device=device).unsqueeze(1)
    wq = torch.tensor(wg, dtype=torch.float32, device=device).unsqueeze(1)
    return xq, wq

XQ, WQ = gauss_legendre_01(PROB.n_quad, DEVICE)

class FourierFeatures(nn.Module):
    def __init__(self, k: int = 8):
        super().__init__()
        freqs = torch.arange(1, k + 1, dtype=torch.float32) * math.pi
        self.register_buffer("freqs", freqs)

    @property
    def out_dim(self) -> int:
        return 1 + 2 * len(self.freqs)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([x, torch.sin(x * self.freqs), torch.cos(x * self.freqs)], dim=-1)

class FourierMLP(nn.Module):
    def __init__(self, hidden: tuple[int, ...], ff_k: int):
        super().__init__()
        self.ff = FourierFeatures(ff_k)
        dims = [self.ff.out_dim] + list(hidden) + [1]
        layers = []
        for i in range(len(dims) - 1):
            lin = nn.Linear(dims[i], dims[i + 1])
            nn.init.xavier_uniform_(lin.weight)
            nn.init.zeros_(lin.bias)
            layers.append(lin)
            if i < len(dims) - 2:
                layers.append(nn.Tanh())
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(self.ff(x))

class HardBCWrapper(nn.Module):
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * (1.0 - x) * self.base(x)

class SineSeriesSource(nn.Module):
    def __init__(self, k_source: int):
        super().__init__()
        self.k_source = k_source
        self.a = nn.Parameter(torch.zeros(k_source, dtype=torch.float32))
        self.register_buffer("kpi", torch.arange(1, k_source + 1, dtype=torch.float32) * math.pi)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        basis = torch.sin(x * self.kpi)
        return (basis * self.a).sum(dim=1, keepdim=True)

def make_test_basis(k_test: int):
    ks = torch.arange(1, k_test + 1, dtype=torch.float32, device=DEVICE).view(-1, 1)
    phi = torch.sin(ks * math.pi * XQ.T)
    dphi = ks * math.pi * torch.cos(ks * math.pi * XQ.T)
    return ks, phi, dphi

def grad_1d(y: torch.Tensor, x: torch.Tensor, create_graph: bool) -> torch.Tensor:
    return torch.autograd.grad(
        y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=create_graph,
        retain_graph=True,
        only_inputs=True,
    )[0]

def d1(model: nn.Module, x_in: torch.Tensor, create_graph: bool = True) -> torch.Tensor:
    x = x_in.detach().clone().requires_grad_(True)
    y = model(x)
    return grad_1d(y, x, create_graph=create_graph)

def d2(model: nn.Module, x_in: torch.Tensor, create_graph: bool = True) -> torch.Tensor:
    x = x_in.detach().clone().requires_grad_(True)
    y = model(x)
    y_x = grad_1d(y, x, create_graph=True)
    y_xx = grad_1d(y_x, x, create_graph=create_graph)
    return y_xx

def source_basis_matrix(x: torch.Tensor, k_source: int) -> torch.Tensor:
    k = torch.arange(1, k_source + 1, dtype=torch.float32, device=x.device).view(1, -1)
    return torch.sin(x * k * math.pi)

def weak_residual_vector(T_model: nn.Module, f_model: nn.Module, model_cfg: ModelConfig) -> torch.Tensor:
    _, phi_test, dphi_test = make_test_basis(model_cfg.k_test)
    dT = d1(T_model, XQ, create_graph=True).T
    f_vals = f_model(XQ).T
    lhs = (dT * dphi_test * WQ.T).sum(dim=1)
    rhs = (f_vals * phi_test * WQ.T).sum(dim=1)
    return lhs - rhs

def weak_loss(T_model: nn.Module, f_model: nn.Module, model_cfg: ModelConfig) -> torch.Tensor:
    return (weak_residual_vector(T_model, f_model, model_cfg) ** 2).mean()

def h1_seminorm_sq(T_model: nn.Module) -> torch.Tensor:
    dT = d1(T_model, XQ, create_graph=True)
    return (WQ * dT**2).sum()

def h2_seminorm_sq(T_model: nn.Module) -> torch.Tensor:
    d2T = d2(T_model, XQ, create_graph=True)
    return (WQ * d2T**2).sum()

def source_coeff_reg(f_model: SineSeriesSource, power: int = 2) -> torch.Tensor:
    k = torch.arange(1, f_model.k_source + 1, dtype=torch.float32, device=f_model.a.device)
    return ((k**power) * f_model.a.pow(2)).mean()

def strong_residual_rms(T_model: nn.Module, f_model: nn.Module, x: torch.Tensor) -> float:
    with torch.enable_grad():
        r = d2(T_model, x, create_graph=False) + f_model(x)
    return float(torch.sqrt((r**2).mean()).detach().cpu().item())

def relative_l2(y_hat: np.ndarray, y_true: np.ndarray) -> float:
    denom = np.sqrt(np.mean(y_true**2)) + 1e-12
    return float(np.sqrt(np.mean((y_hat - y_true)**2)) / denom)


In [ ]:

class SimpleLogger:
    def __init__(self):
        self.history: dict[str, list[float]] = {}

    def push(self, **kwargs: float):
        for k, v in kwargs.items():
            self.history.setdefault(k, []).append(float(v))

def build_models(model_cfg: ModelConfig):
    T_net = HardBCWrapper(FourierMLP(hidden=model_cfg.hidden, ff_k=model_cfg.ff_k)).to(DEVICE)
    f_net = SineSeriesSource(k_source=model_cfg.k_source).to(DEVICE)
    return T_net, f_net

def direct_source_recovery_from_T(T_model: nn.Module, model_cfg: ModelConfig, train_cfg: TrainConfig):
    dT_quad = d1(T_model, XQ, create_graph=False)
    Phi = source_basis_matrix(XQ, model_cfg.k_source)
    k = torch.arange(1, model_cfg.k_source + 1, dtype=torch.float32, device=DEVICE).view(1, -1)
    dPhi = (k * math.pi) * torch.cos(XQ * k * math.pi)
    M = (Phi * WQ).T @ Phi
    b = (dT_quad * dPhi * WQ).sum(dim=0)
    ridge_diag = torch.arange(1, model_cfg.k_source + 1, dtype=torch.float32, device=DEVICE) ** train_cfg.p2_ridge_power
    A = M + train_cfg.p2_ridge * torch.diag(ridge_diag)
    a0 = torch.linalg.solve(A, b)
    stats = {
        "cond_M": float(torch.linalg.cond(M).detach().cpu()),
        "cond_A": float(torch.linalg.cond(A).detach().cpu()),
        "a0_norm_l2": float(torch.norm(a0).detach().cpu()),
        "a0_max_abs": float(torch.max(torch.abs(a0)).detach().cpu()),
    }
    return a0, stats

def train_single_run(problem_cfg: ProblemConfig, model_cfg: ModelConfig, train_cfg: TrainConfig, seed: int, verbose: bool = True):
    set_seed(seed)
    data = make_dataset(problem_cfg, seed)
    T_net, f_net = build_models(model_cfg)
    logger_p1 = SimpleLogger()
    logger_p3 = SimpleLogger()

    best_p1_loss = float("inf")
    best_p1_state = None
    opt1 = optim.Adam(T_net.parameters(), lr=train_cfg.p1_lr)

    t0 = time.time()
    for epoch in range(1, train_cfg.p1_epochs + 1):
        opt1.zero_grad()
        T_pred_obs = T_net(data["x_obs"])
        loss_data = ((T_pred_obs - data["T_obs"])**2).mean()
        loss_h1 = h1_seminorm_sq(T_net)
        loss_h2 = h2_seminorm_sq(T_net)
        loss = train_cfg.p1_w_data * loss_data + train_cfg.p1_w_h1 * loss_h1 + train_cfg.p1_w_h2 * loss_h2
        loss.backward()
        torch.nn.utils.clip_grad_norm_(T_net.parameters(), max_norm=1.0)
        opt1.step()
        logger_p1.push(loss=loss.item(), data=loss_data.item(), h1=loss_h1.item(), h2=loss_h2.item())
        if loss.item() < best_p1_loss:
            best_p1_loss = float(loss.item())
            best_p1_state = {k: v.detach().cpu().clone() for k, v in T_net.state_dict().items()}
        if verbose and (epoch == 1 or epoch % max(500, train_cfg.p1_epochs // 8) == 0 or epoch == train_cfg.p1_epochs):
            print(f"P1 [{epoch:5d}/{train_cfg.p1_epochs:5d}] | loss={loss.item():.3e} | data={loss_data.item():.3e} | h2={loss_h2.item():.3e}")

    if best_p1_state is not None:
        T_net.load_state_dict(best_p1_state)
    phase1_time = time.time() - t0

    a0, p2_stats = direct_source_recovery_from_T(T_net, model_cfg, train_cfg)
    with torch.no_grad():
        f_net.a.copy_(a0)

    a_anchor = a0.detach().clone()
    best_p3_state = None
    best_p3_obj = float("inf")
    opt3 = optim.Adam([
        {"params": T_net.parameters(), "lr": train_cfg.p3_lr_T},
        {"params": f_net.parameters(), "lr": train_cfg.p3_lr_f},
    ])
    all_params = list(T_net.parameters()) + list(f_net.parameters())

    t1 = time.time()
    for epoch in range(1, train_cfg.p3_epochs + 1):
        opt3.zero_grad()
        T_pred_obs = T_net(data["x_obs"])
        loss_data = ((T_pred_obs - data["T_obs"])**2).mean()
        loss_weak = weak_loss(T_net, f_net, model_cfg)
        loss_anchor = ((f_net.a - a_anchor)**2).mean()
        loss_f_reg = source_coeff_reg(f_net, power=2)
        loss_h2 = h2_seminorm_sq(T_net)
        loss = (
            train_cfg.p3_w_data * loss_data
            + train_cfg.p3_w_weak * loss_weak
            + train_cfg.p3_w_anchor * loss_anchor
            + train_cfg.p3_w_f_reg * loss_f_reg
            + train_cfg.p3_w_h2 * loss_h2
        )
        loss.backward()
        grad_norm = float(torch.nn.utils.clip_grad_norm_(all_params, max_norm=train_cfg.p3_grad_clip).detach().cpu())
        opt3.step()
        logger_p3.push(loss=loss.item(), data=loss_data.item(), weak=loss_weak.item(), anchor=loss_anchor.item(), f_reg=loss_f_reg.item(), h2=loss_h2.item(), grad=grad_norm)
        if loss.item() < best_p3_obj:
            best_p3_obj = float(loss.item())
            best_p3_state = {
                "T_net": {k: v.detach().cpu().clone() for k, v in T_net.state_dict().items()},
                "f_net": {k: v.detach().cpu().clone() for k, v in f_net.state_dict().items()},
            }
        if verbose and (epoch == 1 or epoch % max(500, train_cfg.p3_epochs // 10) == 0 or epoch == train_cfg.p3_epochs):
            print(f"P3 [{epoch:5d}/{train_cfg.p3_epochs:5d}] | loss={loss.item():.3e} | data={loss_data.item():.3e} | weak={loss_weak.item():.3e} | anchor={loss_anchor.item():.3e} | grad={grad_norm:.3e}")

    if best_p3_state is not None:
        T_net.load_state_dict(best_p3_state["T_net"])
        f_net.load_state_dict(best_p3_state["f_net"])
    phase3_time = time.time() - t1

    lbfgs_hist = []
    if train_cfg.use_lbfgs:
        params = list(T_net.parameters()) + list(f_net.parameters())
        optimizer_lbfgs = optim.LBFGS(params, lr=train_cfg.lbfgs_lr, max_iter=train_cfg.lbfgs_max_iter_per_step, history_size=50, line_search_fn="strong_wolfe")
        for step in range(1, train_cfg.lbfgs_steps + 1):
            def closure():
                optimizer_lbfgs.zero_grad()
                T_pred_obs = T_net(data["x_obs"])
                loss_data = ((T_pred_obs - data["T_obs"])**2).mean()
                loss_weak = weak_loss(T_net, f_net, model_cfg)
                loss_anchor = ((f_net.a - a_anchor)**2).mean()
                loss_f_reg = source_coeff_reg(f_net, power=2)
                loss_h2 = h2_seminorm_sq(T_net)
                loss = (
                    train_cfg.p3_w_data * loss_data
                    + train_cfg.p3_w_weak * loss_weak
                    + train_cfg.p3_w_anchor * loss_anchor
                    + train_cfg.p3_w_f_reg * loss_f_reg
                    + train_cfg.p3_w_h2 * loss_h2
                )
                loss.backward()
                return loss
            lv = optimizer_lbfgs.step(closure)
            lv = float(lv.detach().cpu().item() if torch.is_tensor(lv) else lv)
            lbfgs_hist.append(lv)
            if verbose and (step == 1 or step % max(10, train_cfg.lbfgs_steps // 10) == 0 or step == train_cfg.lbfgs_steps):
                print(f"LBFGS [{step:4d}/{train_cfg.lbfgs_steps:4d}] | loss={lv:.3e}")

    with torch.no_grad():
        T_eval_np = T_net(data["x_eval"]).detach().cpu().squeeze().numpy()
        f_eval_np = f_net(data["x_eval"]).detach().cpu().squeeze().numpy()

    weak_vec = weak_residual_vector(T_net, f_net, model_cfg).detach().cpu().numpy()
    strong_rms = strong_residual_rms(T_net, f_net, data["x_eval"])

    metrics = {
        "seed": seed,
        "noise_rel": problem_cfg.noise_rel,
        "k_source": model_cfg.k_source,
        "T_l2": float(np.sqrt(np.mean((T_eval_np - data["T_true_eval_np"])**2))),
        "f_l2": float(np.sqrt(np.mean((f_eval_np - data["f_true_eval_np"])**2))),
        "T_rel_l2": relative_l2(T_eval_np, data["T_true_eval_np"]),
        "f_rel_l2": relative_l2(f_eval_np, data["f_true_eval_np"]),
        "T_max_abs": float(np.max(np.abs(T_eval_np - data["T_true_eval_np"]))),
        "f_max_abs": float(np.max(np.abs(f_eval_np - data["f_true_eval_np"]))),
        "weak_rms": float(np.sqrt(np.mean(weak_vec**2))),
        "weak_max_abs": float(np.max(np.abs(weak_vec))),
        "strong_rms": float(strong_rms),
        "phase1_best_loss": float(best_p1_loss),
        "phase3_best_obj": float(best_p3_obj),
        "noise_amp": float(data["noise_amp"]),
        "cond_M": float(p2_stats["cond_M"]),
        "cond_A": float(p2_stats["cond_A"]),
        "phase1_time_s": float(phase1_time),
        "phase3_time_s": float(phase3_time),
        "total_time_s": float(phase1_time + phase3_time),
    }

    artifacts = {
        "data": data,
        "T_eval_np": T_eval_np,
        "f_eval_np": f_eval_np,
        "logger_p1": logger_p1.history,
        "logger_p3": logger_p3.history,
        "lbfgs_hist": lbfgs_hist,
        "weak_vec": weak_vec,
        "source_coeffs": f_net.a.detach().cpu().numpy().copy(),
    }
    return metrics, artifacts


## 1. Cas showcase : reconstruction détaillée sur un run de référence

Cette section produit un cas particulièrement propre avec :
- logs complets,
- figures de reconstruction,
- courbes de losses,
- résidu faible par mode de test,
- coefficients spectraux reconstruits.


In [ ]:

if BENCH.run_showcase:
    metrics_show, art_show = train_single_run(PROB, MODEL, TRAIN, seed=0, verbose=True)
    print(json.dumps(metrics_show, indent=2))
else:
    metrics_show, art_show = None, None


In [ ]:

if metrics_show is not None:
    x_eval_np = art_show["data"]["x_eval_np"]
    T_true_eval_np = art_show["data"]["T_true_eval_np"]
    f_true_eval_np = art_show["data"]["f_true_eval_np"]
    x_obs_np = art_show["data"]["x_obs_np"]
    T_obs_np = art_show["data"]["T_obs_np"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(x_eval_np, T_true_eval_np, label="T true", lw=2.5)
    axes[0].plot(x_eval_np, art_show["T_eval_np"], "--", label="T reconstructed", lw=2.2)
    axes[0].scatter(x_obs_np, T_obs_np, label="observations", zorder=5)
    axes[0].set_title(f"Temperature reconstruction | L2={metrics_show['T_l2']:.3e}")
    axes[0].grid(alpha=0.3)
    axes[0].legend()
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("T(x)")

    axes[1].plot(x_eval_np, f_true_eval_np, label="f true", lw=2.5)
    axes[1].plot(x_eval_np, art_show["f_eval_np"], "--", label="f reconstructed", lw=2.2)
    axes[1].set_title(f"Source reconstruction | L2={metrics_show['f_l2']:.3e}")
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("f(x)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "showcase_reconstruction.png"), dpi=160, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].semilogy(art_show["logger_p1"]["loss"], label="total")
    axes[0].semilogy(art_show["logger_p1"]["data"], label="data")
    axes[0].semilogy(art_show["logger_p1"]["h2"], label="H2")
    axes[0].grid(alpha=0.3)
    axes[0].legend()
    axes[0].set_title("Phase 1 losses")

    axes[1].semilogy(art_show["logger_p3"]["loss"], label="total")
    axes[1].semilogy(art_show["logger_p3"]["data"], label="data")
    axes[1].semilogy(art_show["logger_p3"]["weak"], label="weak")
    axes[1].semilogy(art_show["logger_p3"]["anchor"], label="anchor")
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    axes[1].set_title("Phase 3 losses")

    axes[2].plot(art_show["logger_p3"]["grad"], label="grad norm")
    axes[2].grid(alpha=0.3)
    axes[2].legend()
    axes[2].set_title("Gradient monitor")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "showcase_training_curves.png"), dpi=160, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].stem(np.arange(1, MODEL.k_test + 1), art_show["weak_vec"])
    axes[0].grid(alpha=0.3)
    axes[0].set_title("Weak residual per test mode")
    axes[0].set_xlabel("test mode k")

    axes[1].stem(np.arange(1, MODEL.k_source + 1), art_show["source_coeffs"])
    axes[1].grid(alpha=0.3)
    axes[1].set_title("Recovered source coefficients")
    axes[1].set_xlabel("source mode k")

    err_T = art_show["T_eval_np"] - T_true_eval_np
    err_f = art_show["f_eval_np"] - f_true_eval_np
    axes[2].plot(x_eval_np, err_T, label="T error")
    axes[2].plot(x_eval_np, err_f, label="f error")
    axes[2].axhline(0.0, ls=":")
    axes[2].grid(alpha=0.3)
    axes[2].legend()
    axes[2].set_title("Pointwise errors")
    axes[2].set_xlabel("x")

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "showcase_diagnostics.png"), dpi=160, bbox_inches="tight")
    plt.show()

    if len(art_show["lbfgs_hist"]) > 0:
        plt.figure(figsize=(7, 4))
        plt.semilogy(art_show["lbfgs_hist"])
        plt.grid(alpha=0.3)
        plt.title("LBFGS history")
        plt.xlabel("LBFGS step")
        plt.ylabel("objective")
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "showcase_lbfgs.png"), dpi=160, bbox_inches="tight")
        plt.show()


## 2. Utilitaires pour les benchmarks

On crée maintenant une version allégée de la config d'entraînement afin de pouvoir répéter les expériences sur plusieurs seeds, niveaux de bruit et tailles de base source.


In [ ]:

def make_sweep_train_config(base: TrainConfig, bench_cfg: BenchmarkConfig) -> TrainConfig:
    return TrainConfig(
        p1_epochs=bench_cfg.sweep_p1_epochs,
        p1_lr=base.p1_lr,
        p1_w_data=base.p1_w_data,
        p1_w_h1=base.p1_w_h1,
        p1_w_h2=base.p1_w_h2,
        p2_ridge=base.p2_ridge,
        p2_ridge_power=base.p2_ridge_power,
        p3_epochs=bench_cfg.sweep_p3_epochs,
        p3_lr_T=base.p3_lr_T,
        p3_lr_f=base.p3_lr_f,
        p3_w_data=base.p3_w_data,
        p3_w_weak=base.p3_w_weak,
        p3_w_anchor=base.p3_w_anchor,
        p3_w_f_reg=base.p3_w_f_reg,
        p3_w_h2=base.p3_w_h2,
        p3_grad_clip=base.p3_grad_clip,
        use_lbfgs=base.use_lbfgs,
        lbfgs_steps=bench_cfg.sweep_lbfgs_steps,
        lbfgs_lr=base.lbfgs_lr,
        lbfgs_max_iter_per_step=base.lbfgs_max_iter_per_step,
    )

SWEEP_TRAIN = make_sweep_train_config(TRAIN, BENCH)
print(asdict(SWEEP_TRAIN))


In [ ]:

def run_experiment_grid(seeds, noise_grid, k_source_grid, base_problem_cfg, base_model_cfg, train_cfg, verbose_each=False):
    rows = []
    total = len(seeds) * len(noise_grid) * len(k_source_grid)
    idx = 0
    for noise in noise_grid:
        for k_source in k_source_grid:
            for seed in seeds:
                idx += 1
                print(f"[{idx:3d}/{total:3d}] seed={seed} | noise={noise:.3%} | k_source={k_source}")
                problem_cfg = ProblemConfig(
                    n_obs=base_problem_cfg.n_obs,
                    n_eval=base_problem_cfg.n_eval,
                    n_quad=base_problem_cfg.n_quad,
                    noise_rel=noise,
                )
                model_cfg = ModelConfig(
                    k_test=base_model_cfg.k_test,
                    k_source=k_source,
                    ff_k=base_model_cfg.ff_k,
                    hidden=base_model_cfg.hidden,
                )
                metrics, _ = train_single_run(problem_cfg, model_cfg, train_cfg, seed=seed, verbose=verbose_each)
                rows.append(metrics)
                free_memory()
    return pd.DataFrame(rows)

def summarize_group(df: pd.DataFrame, by_cols: list[str], metric_cols: list[str]) -> pd.DataFrame:
    grouped = df.groupby(by_cols)[metric_cols].agg(["mean", "std", "median", "min", "max"]).reset_index()
    grouped.columns = ["_".join(c).strip("_") for c in grouped.columns.to_flat_index()]
    return grouped


## 3. Étude de robustesse au bruit

On fixe `k_source` à la valeur de référence, puis on répète les runs sur plusieurs seeds et niveaux de bruit.


In [ ]:

noise_df = None
if BENCH.run_noise_study:
    noise_df = run_experiment_grid(
        seeds=BENCH.seeds,
        noise_grid=BENCH.noise_grid,
        k_source_grid=(MODEL.k_source,),
        base_problem_cfg=PROB,
        base_model_cfg=MODEL,
        train_cfg=SWEEP_TRAIN,
        verbose_each=False,
    )
    noise_df.to_csv(os.path.join(TABLE_DIR, "noise_study_runs.csv"), index=False)
    print(noise_df.head())


In [ ]:

if noise_df is not None and len(noise_df) > 0:
    metric_cols = ["T_l2", "f_l2", "T_rel_l2", "f_rel_l2", "weak_rms", "strong_rms"]
    noise_summary = summarize_group(noise_df, by_cols=["noise_rel"], metric_cols=metric_cols)
    noise_summary.to_csv(os.path.join(TABLE_DIR, "noise_study_summary.csv"), index=False)
    print(noise_summary)

    grouped = noise_df.groupby("noise_rel")
    noise_vals = sorted(noise_df["noise_rel"].unique())
    f_mean = [grouped.get_group(n)["f_l2"].mean() for n in noise_vals]
    f_std = [grouped.get_group(n)["f_l2"].std(ddof=1) if len(grouped.get_group(n)) > 1 else 0.0 for n in noise_vals]
    T_mean = [grouped.get_group(n)["T_l2"].mean() for n in noise_vals]
    T_std = [grouped.get_group(n)["T_l2"].std(ddof=1) if len(grouped.get_group(n)) > 1 else 0.0 for n in noise_vals]

    plt.figure(figsize=(8, 4))
    plt.errorbar([100*n for n in noise_vals], f_mean, yerr=f_std, marker="o", capsize=4, label="f L2")
    plt.errorbar([100*n for n in noise_vals], T_mean, yerr=T_std, marker="s", capsize=4, label="T L2")
    plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.xlabel("Noise level (%)")
    plt.ylabel("Mean error ± std")
    plt.title("Noise robustness")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "noise_robustness_errorbars.png"), dpi=160, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 4))
    for noise in noise_vals:
        subset = noise_df[noise_df["noise_rel"] == noise]
        plt.scatter([100*noise]*len(subset), subset["f_l2"], s=40)
    plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.xlabel("Noise level (%)")
    plt.ylabel("Per-run source L2 error")
    plt.title("Noise robustness — seed dispersion")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "noise_robustness_scatter.png"), dpi=160, bbox_inches="tight")
    plt.show()


## 4. Sweep sur `k_source`

On fixe le bruit, puis on balaye différentes tailles de base pour la source. C’est l’étude la plus importante pour démontrer le rôle du contrôle de complexité spectrale.


In [ ]:

ks_df = None
if BENCH.run_k_source_sweep:
    ks_df = run_experiment_grid(
        seeds=BENCH.seeds,
        noise_grid=(PROB.noise_rel,),
        k_source_grid=BENCH.k_source_grid,
        base_problem_cfg=PROB,
        base_model_cfg=MODEL,
        train_cfg=SWEEP_TRAIN,
        verbose_each=False,
    )
    ks_df.to_csv(os.path.join(TABLE_DIR, "k_source_sweep_runs.csv"), index=False)
    print(ks_df.head())


In [ ]:

if ks_df is not None and len(ks_df) > 0:
    metric_cols = ["T_l2", "f_l2", "T_rel_l2", "f_rel_l2", "weak_rms", "strong_rms"]
    ks_summary = summarize_group(ks_df, by_cols=["k_source"], metric_cols=metric_cols)
    ks_summary.to_csv(os.path.join(TABLE_DIR, "k_source_sweep_summary.csv"), index=False)
    print(ks_summary)

    grouped = ks_df.groupby("k_source")
    k_vals = sorted(ks_df["k_source"].unique())
    f_mean = [grouped.get_group(k)["f_l2"].mean() for k in k_vals]
    f_std = [grouped.get_group(k)["f_l2"].std(ddof=1) if len(grouped.get_group(k)) > 1 else 0.0 for k in k_vals]
    T_mean = [grouped.get_group(k)["T_l2"].mean() for k in k_vals]
    T_std = [grouped.get_group(k)["T_l2"].std(ddof=1) if len(grouped.get_group(k)) > 1 else 0.0 for k in k_vals]

    plt.figure(figsize=(8, 4))
    plt.errorbar(k_vals, f_mean, yerr=f_std, marker="o", capsize=4, label="f L2")
    plt.errorbar(k_vals, T_mean, yerr=T_std, marker="s", capsize=4, label="T L2")
    plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.xlabel("k_source")
    plt.ylabel("Mean error ± std")
    plt.title("Sensitivity to source basis size")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "k_source_sensitivity_errorbars.png"), dpi=160, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.boxplot([ks_df[ks_df["k_source"] == k]["f_l2"].values for k in k_vals], tick_labels=k_vals)
    plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.xlabel("k_source")
    plt.ylabel("Source L2 error")
    plt.title("k_source sweep — distribution over seeds")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "k_source_sensitivity_boxplot.png"), dpi=160, bbox_inches="tight")
    plt.show()


## 5. Grille complète bruit × `k_source` × seed

Cette partie fournit une vue globale de la robustesse du pipeline et permet de produire des heatmaps. Elle est particulièrement utile pour un papier.


In [ ]:

full_df = None
if BENCH.run_full_grid:
    full_df = run_experiment_grid(
        seeds=BENCH.seeds,
        noise_grid=BENCH.noise_grid,
        k_source_grid=BENCH.k_source_grid,
        base_problem_cfg=PROB,
        base_model_cfg=MODEL,
        train_cfg=SWEEP_TRAIN,
        verbose_each=False,
    )
    full_df.to_csv(os.path.join(TABLE_DIR, "full_grid_runs.csv"), index=False)
    print(full_df.shape)
    print(full_df.head())


In [ ]:

if full_df is not None and len(full_df) > 0:
    full_summary = summarize_group(
        full_df,
        by_cols=["noise_rel", "k_source"],
        metric_cols=["T_l2", "f_l2", "T_rel_l2", "f_rel_l2", "weak_rms", "strong_rms"]
    )
    full_summary.to_csv(os.path.join(TABLE_DIR, "full_grid_summary.csv"), index=False)
    print(full_summary.head())

    pivot_f = full_df.pivot_table(index="noise_rel", columns="k_source", values="f_l2", aggfunc="mean")
    pivot_T = full_df.pivot_table(index="noise_rel", columns="k_source", values="T_l2", aggfunc="mean")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    im0 = axes[0].imshow(np.log10(pivot_f.values + 1e-16), aspect="auto", origin="lower")
    axes[0].set_xticks(range(len(pivot_f.columns)))
    axes[0].set_xticklabels(pivot_f.columns)
    axes[0].set_yticks(range(len(pivot_f.index)))
    axes[0].set_yticklabels([f"{100*n:.1f}%" for n in pivot_f.index])
    axes[0].set_xlabel("k_source")
    axes[0].set_ylabel("noise level")
    axes[0].set_title("log10 mean source L2 error")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(np.log10(pivot_T.values + 1e-16), aspect="auto", origin="lower")
    axes[1].set_xticks(range(len(pivot_T.columns)))
    axes[1].set_xticklabels(pivot_T.columns)
    axes[1].set_yticks(range(len(pivot_T.index)))
    axes[1].set_yticklabels([f"{100*n:.1f}%" for n in pivot_T.index])
    axes[1].set_xlabel("k_source")
    axes[1].set_ylabel("noise level")
    axes[1].set_title("log10 mean temperature L2 error")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "heatmaps_noise_ksource.png"), dpi=160, bbox_inches="tight")
    plt.show()


## 6. Reconstructions représentatives pour plusieurs régimes

On montre quelques cas concrets pour illustrer visuellement :
- bruit faible / `k_source` optimal,
- bruit modéré / `k_source` optimal,
- bruit plus fort / `k_source` surdimensionné.


In [ ]:

representative_cases = [
    {"seed": 0, "noise_rel": 0.00, "k_source": 3},
    {"seed": 0, "noise_rel": 0.01, "k_source": 3},
    {"seed": 0, "noise_rel": 0.05, "k_source": 3},
    {"seed": 0, "noise_rel": 0.01, "k_source": 6},
]

case_results = []
for case in representative_cases:
    print("Running representative case:", case)
    p = ProblemConfig(n_obs=PROB.n_obs, n_eval=PROB.n_eval, n_quad=PROB.n_quad, noise_rel=case["noise_rel"])
    m = ModelConfig(k_test=MODEL.k_test, k_source=case["k_source"], ff_k=MODEL.ff_k, hidden=MODEL.hidden)
    met, art = train_single_run(p, m, SWEEP_TRAIN, seed=case["seed"], verbose=False)
    case_results.append((case, met, art))
    free_memory()

fig, axes = plt.subplots(len(case_results), 2, figsize=(14, 4 * len(case_results)))
if len(case_results) == 1:
    axes = np.array([axes])

for row, (case, met, art) in enumerate(case_results):
    x = art["data"]["x_eval_np"]
    x_obs = art["data"]["x_obs_np"]
    T_obs = art["data"]["T_obs_np"]

    axes[row, 0].plot(x, art["data"]["T_true_eval_np"], label="T true", lw=2.3)
    axes[row, 0].plot(x, art["T_eval_np"], "--", label="T rec", lw=2.0)
    axes[row, 0].scatter(x_obs, T_obs, s=24, label="obs", zorder=5)
    axes[row, 0].grid(alpha=0.3)
    axes[row, 0].set_title(f"T | noise={case['noise_rel']:.1%}, k_source={case['k_source']}, L2={met['T_l2']:.2e}")
    axes[row, 0].legend()

    axes[row, 1].plot(x, art["data"]["f_true_eval_np"], label="f true", lw=2.3)
    axes[row, 1].plot(x, art["f_eval_np"], "--", label="f rec", lw=2.0)
    axes[row, 1].grid(alpha=0.3)
    axes[row, 1].set_title(f"f | noise={case['noise_rel']:.1%}, k_source={case['k_source']}, L2={met['f_l2']:.2e}")
    axes[row, 1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "representative_cases.png"), dpi=160, bbox_inches="tight")
plt.show()


## 7. Tables récapitulatives “article-ready”

On génère des tableaux compacts avec moyenne ± écart-type, utiles pour une section résultats.


In [ ]:

def format_mean_std(df: pd.DataFrame, group_col: str, metric: str) -> pd.DataFrame:
    g = df.groupby(group_col)[metric].agg(["mean", "std", "median", "min", "max"]).reset_index()
    g["mean±std"] = g.apply(lambda r: f"{r['mean']:.3e} ± {r['std']:.3e}", axis=1)
    return g[[group_col, "mean±std", "median", "min", "max"]]

if noise_df is not None and len(noise_df) > 0:
    table_noise_f = format_mean_std(noise_df, "noise_rel", "f_l2")
    table_noise_T = format_mean_std(noise_df, "noise_rel", "T_l2")
    print(table_noise_f)
    print(table_noise_T)
    table_noise_f.to_csv(os.path.join(TABLE_DIR, "article_table_noise_f.csv"), index=False)
    table_noise_T.to_csv(os.path.join(TABLE_DIR, "article_table_noise_T.csv"), index=False)

if ks_df is not None and len(ks_df) > 0:
    table_k_f = format_mean_std(ks_df, "k_source", "f_l2")
    table_k_T = format_mean_std(ks_df, "k_source", "T_l2")
    print(table_k_f)
    print(table_k_T)
    table_k_f.to_csv(os.path.join(TABLE_DIR, "article_table_ksource_f.csv"), index=False)
    table_k_T.to_csv(os.path.join(TABLE_DIR, "article_table_ksource_T.csv"), index=False)


## 8. Conclusion expérimentale

Ce qu’on s’attend à observer dans les résultats :

- `T(x)` est généralement très bien reconstruit pour toute la plage de bruit testée ;
- la reconstruction de `f(x)` est nettement plus sensible à la complexité spectrale ;
- un `k_source` trop grand dégrade la robustesse en présence de bruit ;
- il existe souvent un sweet spot autour de `k_source = 3` ;
- la formulation faible VPINN reste très stable sur le problème inverse 1D.

Cette section, accompagnée des figures et tableaux générés ci-dessus, est déjà proche d’un niveau “article”.


In [ ]:

manifest = {
    "problem_config": asdict(PROB),
    "model_config": asdict(MODEL),
    "train_config": asdict(TRAIN),
    "benchmark_config": asdict(BENCH),
    "device": str(DEVICE),
    "generated_files": sorted([
        os.path.join(dp, f)
        for dp, dn, fn in os.walk(ROOT_DIR)
        for f in fn
    ]),
}
with open(os.path.join(ROOT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Generated files:")
for p in manifest["generated_files"]:
    print(" -", p)
